In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import Ridge

In [3]:
df=pd.read_csv('final_credit_dataset.csv')

In [5]:
df.head(2)

,2,Age,Work_Experience,Projects_Last_1M,Projects_Last_6M,Upcoming_Projects,Avg_Project_Value,Monthly_Expenses,Income_1M,Existing_Loans,...,Platform,monthly_Cash_Inflow,monthly_Cash_Outflow,Savings,EMI_Income_Ratio,Income_Stability,Experience_Score,Loan_Burden,Income_per_Project,Synthetic_Score
0,0,19,0,0,3,0,33799,0,0,0,...,Upwork,0,0,0,0.0,0.500000,0.000000,0,0.000,422
1,1,38,19,7,31,7,22633,87137,158431,0,...,Upwork,158431,87137,71294,0.0,5.166667,0.487179,0,19803.875,850


In [6]:
df.columns

Index(['2', 'Age', 'Work_Experience', 'Projects_Last_1M', 'Projects_Last_6M',
       'Upcoming_Projects', 'Avg_Project_Value', 'Monthly_Expenses',
       'Income_1M', 'Existing_Loans', 'EMI_Amount', 'Freelancer_Category',
       'Platform', 'monthly_Cash_Inflow', 'monthly_Cash_Outflow', 'Savings',
       'EMI_Income_Ratio', 'Income_Stability', 'Experience_Score',
       'Loan_Burden', 'Income_per_Project', 'Synthetic_Score'],
      dtype='object')

In [7]:
df = df.drop(columns=['2'])
df.head()

,Age,Work_Experience,Projects_Last_1M,Projects_Last_6M,Upcoming_Projects,Avg_Project_Value,Monthly_Expenses,Income_1M,Existing_Loans,EMI_Amount,...,Platform,monthly_Cash_Inflow,monthly_Cash_Outflow,Savings,EMI_Income_Ratio,Income_Stability,Experience_Score,Loan_Burden,Income_per_Project,Synthetic_Score
0,19,0,0,3,0,33799,0,0,0,0,...,Upwork,0,0,0,0.000000,0.500000,0.000000,0,0.000000,422
1,38,19,7,31,7,22633,87137,158431,0,0,...,Upwork,158431,87137,71294,0.000000,5.166667,0.487179,0,19803.875000,850
2,33,14,8,24,5,9485,60325,75880,3,10209,...,Upwork,75880,70534,5346,0.134540,4.000000,0.411765,30627,8431.111111,580
3,38,18,8,22,7,45077,212763,360616,0,0,...,Kaggle,360616,212763,147853,0.000000,3.666667,0.461538,0,40068.444440,850
4,54,35,3,18,2,35847,64525,107541,2,17148,...,Spotify,107541,81673,25868,0.159454,3.000000,0.636364,34296,26885.250000,549


In [8]:
y = df["Synthetic_Score"]
X = df.drop(columns=["Synthetic_Score"])

print(X.shape, y.shape)

(50000, 20) (50000,)


In [9]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical Columns:\n", num_cols)
print("\nCategorical Columns:\n", cat_cols)

Numerical Columns:
 ['Age', 'Work_Experience', 'Projects_Last_1M', 'Projects_Last_6M', 'Upcoming_Projects', 'Avg_Project_Value', 'Monthly_Expenses', 'Income_1M', 'Existing_Loans', 'EMI_Amount', 'monthly_Cash_Inflow', 'monthly_Cash_Outflow', 'Savings', 'EMI_Income_Ratio', 'Income_Stability', 'Experience_Score', 'Loan_Burden', 'Income_per_Project']

Categorical Columns:
 ['Freelancer_Category', 'Platform']


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(40000, 20) (10000, 20)


In [11]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import numpy as np

scaler = StandardScaler()
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Numerical
X_train_num = scaler.fit_transform(X_train[num_cols])
X_test_num = scaler.transform(X_test[num_cols])

# Categorical
X_train_cat = encoder.fit_transform(X_train[cat_cols])
X_test_cat = encoder.transform(X_test[cat_cols])

# Combine
X_train_final = np.concatenate([X_train_num, X_train_cat], axis=1)
X_test_final = np.concatenate([X_test_num, X_test_cat], axis=1)

print(X_train_final.shape, X_test_final.shape)

(40000, 54) (10000, 54)


In [12]:
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np

model = Ridge(alpha=0.1)
model.fit(X_train_final, y_train)

y_pred = model.predict(X_test_final)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("R2:", r2)
print("MAE:", mae)
print("RMSE:", rmse)

R2: 0.9035273903028676
MAE: 37.71440312894913
RMSE: 49.58135368105635


In [13]:
import joblib
import os

os.makedirs("model", exist_ok=True)

joblib.dump(model, "model/model.pkl")
joblib.dump(scaler, "model/scaler.pkl")
joblib.dump(encoder, "model/encoder.pkl")
joblib.dump(num_cols, "model/num_cols.pkl")
joblib.dump(cat_cols, "model/cat_cols.pkl")

print("All files saved successfully")

All files saved successfully
